# Step 6 smoke test (BF16)

Thin wrapper: all logic lives in the repo (`ptlegal_eval.smoke`). Five **dev** questions per model plus one worst-case prompt at the 7,500-character budget, all in BF16, to
confirm loading, chat templates and memory. This is not a result.

Before running: **Accelerator = GPU T4 x2**, **Internet = on**, and add your private dataset that
holds `chunks.jsonl` (the law text, kept out of git) under *Input*. Then *Run All* (about 35 min;
keep the tab open). It stops early, with a plain message, if the hardware is wrong.

In [ ]:
# 1. Setup (about 3 min): latest code from GitHub
%cd /kaggle/working
!pip -q install -U "transformers>=5.10" accelerate huggingface_hub
!rm -rf pt-legal-rag-eval && git clone -q -b step-6-runner https://github.com/cardohso/pt-legal-rag-eval.git
%cd pt-legal-rag-eval
!pip -q install -e .
!git rev-parse HEAD

In [ ]:
# 2. Hardware gate: stops here, before any download, unless this session has two T4 GPUs
import subprocess
import sys

check = subprocess.run(
    [sys.executable, "-m", "ptlegal_eval.gpucheck"], capture_output=True, text=True
)
print(check.stdout + check.stderr)
if check.returncode != 0:
    raise RuntimeError(
        "Stopped before any download. Fix what is printed above (wrong GPU, or the setup "
        "failed: check Internet = on), then Run All again."
    )

In [ ]:
# 3. Find the chunks file you added as input (searches any depth)
import glob
import os

found = glob.glob("/kaggle/input/**/chunks.jsonl", recursive=True)
print(found)
if len(found) != 1:
    for root, _dirs, files in os.walk("/kaggle/input"):
        print(root, files[:5])
    raise RuntimeError("expected one chunks.jsonl under /kaggle/input: add the dataset as Input")
CHUNKS = found[0]

In [ ]:
# 4. Run the smoke test: verifies the chunks against the committed hashes, then each model in turn
!python -m ptlegal_eval.smoke run --chunks {CHUNKS} --out /kaggle/working/smoke_bf16_7500

In [ ]:
# 5. Summary, and one zip to download from the Output panel
!python -m ptlegal_eval.smoke report /kaggle/working/smoke_bf16_7500
!cd /kaggle/working && zip -rq smoke_bf16_7500.zip smoke_bf16_7500
!ls -la /kaggle/working/smoke_bf16_7500.zip /kaggle/working/smoke_bf16_7500

Download `smoke_bf16_7500.zip` from the **Output** panel (right side, *Data*) and tell Claude where you saved it.